# MaNGA drpall HEALPix collision check
Analyze the *actual* catalog used by MMU (`drpall-v3_1_1.fits` MANGA HDU, optionally filtered by DAPDONE) to find the healpix order that resolves all row-level collisions.

In [1]:
from collections import Counter

import healpy as hp
import numpy as np
from astropy.table import Table

FITS = '/mnt/home/thehir/tmp-catalog-downloads/drpall-v3_1_1.fits'

## Load the MANGA HDU and inspect columns

In [2]:
cat = Table.read(FITS, hdu='MANGA')
print(f'n_rows = {len(cat)}')
ra_candidates = [c for c in cat.colnames if c.lower() in ('objra', 'ifura', 'ra', 'catalog_ra')]
dec_candidates = [c for c in cat.colnames if c.lower() in ('objdec', 'ifudec', 'dec', 'catalog_dec')]
id_candidates = [c for c in cat.colnames if c.lower() in ('mangaid', 'plateifu', 'plate', 'ifudsgn')]
print(f'ra columns: {ra_candidates}')
print(f'dec columns: {dec_candidates}')
print(f'id columns: {id_candidates}')

n_rows = 11273
ra columns: ['objra', 'ifura']
dec columns: ['objdec', 'ifudec']
id columns: ['plate', 'ifudsgn', 'plateifu', 'mangaid']


In [3]:
# MMU uses objra/objdec via build_parent_sample.py. Stick with that.
RA_COL = 'objra' if 'objra' in cat.colnames else ra_candidates[0]
DEC_COL = 'objdec' if 'objdec' in cat.colnames else dec_candidates[0]
print(f'using {RA_COL}/{DEC_COL}')

ra = np.asarray(cat[RA_COL], dtype=np.float64)
dec = np.asarray(cat[DEC_COL], dtype=np.float64)

finite = np.isfinite(ra) & np.isfinite(dec)
print(f'finite coords: {finite.sum()}/{len(ra)}')
ra, dec = ra[finite], dec[finite]

using objra/objdec
finite coords: 11273/11273


## How many unique galaxies vs observations?
MaNGA reobserves galaxies — same `mangaid` can produce multiple `plateifu` rows at identical (objra, objdec).

In [4]:
if 'mangaid' in cat.colnames:
    mangaids = [x.decode() if isinstance(x, bytes) else str(x) for x in cat['mangaid'][finite]]
    print(f'unique mangaid: {len(set(mangaids))}')
if 'plateifu' in cat.colnames:
    plateifus = [x.decode() if isinstance(x, bytes) else str(x) for x in cat['plateifu'][finite]]
    print(f'unique plateifu: {len(set(plateifus))}')

coord_counts = Counter(zip(ra.tolist(), dec.tolist()))
dup_coords = [(r, d, n) for (r, d), n in coord_counts.items() if n > 1]
print(f'bit-identical (ra, dec) groups: {len(dup_coords)}')
print(f'total rows in identical groups: {sum(n for _, _, n in dup_coords)}')
print(f'max rows at one coordinate: {max((n for _, _, n in dup_coords), default=0)}')

unique mangaid: 10632
unique plateifu: 11273
bit-identical (ra, dec) groups: 169
total rows in identical groups: 828
max rows at one coordinate: 49


## Healpix collision sweep

In [ ]:
for order in range(5, 31):
    nside = 2 ** order
    pix = hp.ang2pix(nside, ra, dec, lonlat=True, nest=True)
    _, counts = np.unique(pix, return_counts=True)
    print(f'order={order:2d} max_per_pixel={counts.max():5d} n_collisions={(counts>1).sum()}')

order= 5 max_per_pixel=  810 n_collisions=1043
order= 6 max_per_pixel=  456 n_collisions=2290
order= 7 max_per_pixel=  316 n_collisions=2289
order= 8 max_per_pixel=  117 n_collisions=1428
order= 9 max_per_pixel=  117 n_collisions=746
order=10 max_per_pixel=   62 n_collisions=396
order=11 max_per_pixel=   49 n_collisions=256
order=12 max_per_pixel=   49 n_collisions=277
order=13 max_per_pixel=   49 n_collisions=234
order=14 max_per_pixel=   49 n_collisions=179
order=15 max_per_pixel=   49 n_collisions=175
order=16 max_per_pixel=   49 n_collisions=173
order=17 max_per_pixel=   49 n_collisions=172
order=18 max_per_pixel=   49 n_collisions=170
order=19 max_per_pixel=   49 n_collisions=169
order=20 max_per_pixel=   49 n_collisions=168
order=21 max_per_pixel=   49 n_collisions=167
order=22 max_per_pixel=   49 n_collisions=165
order=23 max_per_pixel=   49 n_collisions=165
order=24 max_per_pixel=   49 n_collisions=166
order=25 max_per_pixel=   49 n_collisions=169
order=26 max_per_pixel=   49 n

## Inspect the bit-identical duplicates
If any remain at infinite order, list their mangaid/plateifu to see whether reobservations explain them.

In [6]:
ra_all = np.asarray(cat[RA_COL], dtype=np.float64)
dec_all = np.asarray(cat[DEC_COL], dtype=np.float64)
mangaid_all = cat['mangaid'] if 'mangaid' in cat.colnames else None
plateifu_all = cat['plateifu'] if 'plateifu' in cat.colnames else None

for r, d, n in dup_coords:
    idx = np.where((ra_all == r) & (dec_all == d))[0]
    print(f'\nra={r:.6f} dec={d:.6f} count={n} rows={list(idx)}')
    for i in idx:
        mid = mangaid_all[i] if mangaid_all is not None else ''
        pid = plateifu_all[i] if plateifu_all is not None else ''
        if isinstance(mid, bytes): mid = mid.decode()
        if isinstance(pid, bytes): pid = pid.decode()
        print(f'  row={i} mangaid={mid!r} plateifu={pid!r}')


ra=56.702338 dec=68.096625 count=2 rows=[np.int64(17), np.int64(10663)]
  row=17 mangaid='51-100' plateifu='10141-12701'
  row=10663 mangaid='51-49' plateifu='9675-12703'

ra=57.665758 dec=68.070567 count=42 rows=[np.int64(22), np.int64(39), np.int64(56), np.int64(73), np.int64(90), np.int64(107), np.int64(124), np.int64(141), np.int64(158), np.int64(175), np.int64(427), np.int64(444), np.int64(503), np.int64(562), np.int64(578), np.int64(595), np.int64(2951), np.int64(2982), np.int64(2999), np.int64(3016), np.int64(3033), np.int64(3050), np.int64(3067), np.int64(3084), np.int64(3101), np.int64(3118), np.int64(3135), np.int64(3152), np.int64(3169), np.int64(3186), np.int64(3203), np.int64(3220), np.int64(3237), np.int64(3254), np.int64(3271), np.int64(3288), np.int64(3305), np.int64(3322), np.int64(3339), np.int64(10633), np.int64(10650), np.int64(10666)]
  row=22 mangaid='51-70' plateifu='10141-1901'
  row=39 mangaid='51-70' plateifu='10142-1901'
  row=56 mangaid='51-70' plateifu='10

## Cross-check against HDF5 histogram
The staging histogram has 10,735 rows at order 10. Does filtering drpall reproduce that count?

In [7]:
HIST = '/mnt/ceph/users/polymathic/manga-staging-tmp/mmu_manga/intermediate/mmu_manga/intermediate/row_count_mapping_histogram.npz'
with open(HIST, 'rb') as f:
    full = np.frombuffer(f.read(), dtype=np.int64)
print(f'hdf5 rows (histogram): {full.sum()}')
print(f'drpall MANGA rows: {len(cat)}')
print(f'drpall MANGA rows w/ finite coords: {finite.sum()}')
print(f'diff: drpall - hdf5 = {finite.sum() - full.sum()}')

hdf5 rows (histogram): 10735
drpall MANGA rows: 11273
drpall MANGA rows w/ finite coords: 11273
diff: drpall - hdf5 = 538
